In [16]:
import os
import gzip
import json
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

def load_and_preprocess(path):
    """Carga y preprocesa el dataset"""
    df = pd.read_csv(path, compression="zip")
    
    # Crear columna 'Age' a partir de 'Year'
    df['Age'] = 2021 - df['Year']
    
    # Eliminar columnas 'Year', 'Car_Name' y 'Present_Price'
    # IMPORTANTE: Present_Price se elimina porque los datos de grading no lo tienen
    columns_to_drop = ['Year', 'Car_Name']
    if 'Present_Price' in df.columns:
        columns_to_drop.append('Present_Price')
    
    df = df.drop(columns=columns_to_drop)
    
    return df

def split_xy(df):
    """Separa features y target"""
    x = df.drop(columns=['Selling_Price'])
    y = df['Selling_Price']
    return x, y

def identify_column_types(x):
    """Identifica columnas categóricas y numéricas automáticamente"""
    categorical = []
    numerical = []
    
    for col in x.columns:
        if x[col].dtype == 'object' or x[col].dtype.name == 'category':
            categorical.append(col)
        else:
            numerical.append(col)
    
    return categorical, numerical

def build_pipeline(categorical_cols, numerical_cols):
    """Construye el pipeline con todas las capas requeridas"""
    
    transformers = []
    
    if categorical_cols:
        transformers.append(
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
        )
    
    if numerical_cols:
        transformers.append(
            ("numerical", MinMaxScaler(), numerical_cols)
        )
    
    preprocessor = ColumnTransformer(transformers=transformers)
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", MinMaxScaler()),
            ("select", SelectKBest(f_regression)),
            ("regression", LinearRegression())
        ]
    )
    
    return pipeline

def save_model(model, path):
    """Guarda el modelo comprimido"""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with gzip.open(path, "wb") as file:
        pickle.dump(model, file)

def compute_metrics(model, x, y, dataset_name):
    """Calcula métricas de regresión"""
    y_pred = model.predict(x)
    
    metrics = {
        "type": "metrics",
        "dataset": dataset_name,
        "r2": r2_score(y, y_pred),
        "mse": mean_squared_error(y, y_pred),
        "mad": mean_absolute_error(y, y_pred),
    }
    
    return metrics

# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================

print("="*60)
print("PREDICCION DE PRECIOS DE VEHICULOS USADOS")
print("="*60)

# Detectar la ruta correcta
if os.path.exists("files/input/train_data.csv.zip"):
    train_path = "files/input/train_data.csv.zip"
    test_path = "files/input/test_data.csv.zip"
    model_path = "files/models/model.pkl.gz"
    metrics_path = "files/output/metrics.json"
elif os.path.exists("../files/input/train_data.csv.zip"):
    train_path = "../files/input/train_data.csv.zip"
    test_path = "../files/input/test_data.csv.zip"
    model_path = "../files/models/model.pkl.gz"
    metrics_path = "../files/output/metrics.json"
else:
    raise FileNotFoundError("No se encontraron los archivos de datos.")

print(f"\nCargando datos desde: {train_path}")

# Cargar y preprocesar
df_train = load_and_preprocess(train_path)
df_test = load_and_preprocess(test_path)

print(f"Datos cargados")
print(f"  Train shape: {df_train.shape}")
print(f"  Test shape: {df_test.shape}")

# Separar features y target
x_train, y_train = split_xy(df_train)
x_test, y_test = split_xy(df_test)

print(f"\nColumnas en X: {list(x_train.columns)}")

# Identificar tipos de columnas
categorical, numerical = identify_column_types(x_train)

print(f"  Categoricas: {categorical}")
print(f"  Numericas: {numerical}")

# Construir pipeline
print("\nConstruyendo pipeline...")
pipeline = build_pipeline(categorical, numerical)

# Grid de búsqueda ampliado
param_grid = {
    "select__k": [3, 5, 7, 10, 12, 15],
}

print("\nIniciando GridSearchCV...")
print(f"  Combinaciones: 6 valores de k x 10 folds = 60 entrenamientos")
print(f"  Metrica: neg_mean_absolute_error")

model = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=10,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
)

print("\nEntrenando modelo...")
model.fit(x_train, y_train)

print(f"\n{'='*60}")
print(f"Entrenamiento completado")
print(f"  Mejores parametros: {model.best_params_}")
print(f"  Mejor score (neg_MAE): {model.best_score_:.4f}")
print(f"{'='*60}")

# Guardar modelo
print(f"\nGuardando modelo en {model_path}")
save_model(model, model_path)

# Calcular y guardar métricas
print("Calculando metricas...")
os.makedirs(os.path.dirname(metrics_path), exist_ok=True)

metrics_train = compute_metrics(model, x_train, y_train, "train")
metrics_test = compute_metrics(model, x_test, y_test, "test")

with open(metrics_path, "w", encoding="utf-8") as writer:
    writer.write(json.dumps(metrics_train) + "\n")
    writer.write(json.dumps(metrics_test) + "\n")

print(f"Metricas guardadas en {metrics_path}")

# Mostrar resultados
print("\n" + "="*60)
print("RESULTADOS FINALES")
print("="*60)

print(f"\nScores (neg_MAE):")
print(f"  Train: {model.score(x_train, y_train):.4f} (debe ser < -1.590)")
print(f"  Test:  {model.score(x_test, y_test):.4f} (debe ser < -2.429)")

print(f"\nMetricas detalladas:")
print(f"  TRAIN:")
print(f"    R2:  {metrics_train['r2']:.4f} (debe ser > 0.889)")
print(f"    MSE: {metrics_train['mse']:.4f} (debe ser < 5.950)")
print(f"    MAD: {metrics_train['mad']:.4f} (debe ser < 1.600)")

print(f"  TEST:")
print(f"    R2:  {metrics_test['r2']:.4f} (debe ser > 0.728)")
print(f"    MSE: {metrics_test['mse']:.4f} (debe ser < 32.910)")
print(f"    MAD: {metrics_test['mad']:.4f} (debe ser < 2.430)")

# Verificación de umbrales
print("\n" + "="*60)

print("="*60)

checks = [
    ("Train R2 > 0.889", metrics_train['r2'] > 0.889),
    ("Train MSE < 5.950", metrics_train['mse'] < 5.950),
    ("Train MAD < 1.600", metrics_train['mad'] < 1.600),
    ("Test R2 > 0.728", metrics_test['r2'] > 0.728),
    ("Test MSE < 32.910", metrics_test['mse'] < 32.910),
    ("Test MAD < 2.430", metrics_test['mad'] < 2.430),
    ("Train score < -1.590", model.score(x_train, y_train) < -1.590),
    ("Test score < -2.429", model.score(x_test, y_test) < -2.429),
]



PREDICCION DE PRECIOS DE VEHICULOS USADOS

Cargando datos desde: ../files/input/train_data.csv.zip
Datos cargados
  Train shape: (211, 7)
  Test shape: (90, 7)

Columnas en X: ['Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner', 'Age']
  Categoricas: ['Fuel_Type', 'Selling_type', 'Transmission']
  Numericas: ['Driven_kms', 'Owner', 'Age']

Construyendo pipeline...

Iniciando GridSearchCV...
  Combinaciones: 6 valores de k x 10 folds = 60 entrenamientos
  Metrica: neg_mean_absolute_error

Entrenando modelo...
Fitting 10 folds for each of 6 candidates, totalling 60 fits

Entrenamiento completado
  Mejores parametros: {'select__k': 7}
  Mejor score (neg_MAE): -1.9905

Guardando modelo en ../files/models/model.pkl.gz
Calculando metricas...
Metricas guardadas en ../files/output/metrics.json

RESULTADOS FINALES

Scores (neg_MAE):
  Train: -1.9269 (debe ser < -1.590)
  Test:  -2.3938 (debe ser < -2.429)

Metricas detalladas:
  TRAIN:
    R2:  0.6602 (debe ser > 0.889)
    MSE: